# 02 — Loading Documents: TXT, Markdown, PDFs, Folders & Web Pages

**By the end of this notebook, you can convert diverse file formats into clean LangChain Documents while handling encodings, empty pages, and web safety.**

### Learning Objectives
- Understand `Document(page_content, metadata)` as LangChain's universal data envelope.
- Load `.txt`, `.md`, multi-page `.pdf`, whole folders, and HTML web pages (with safe offline fallback).
- Handle common traps: UTF-8 international characters, filenames with spaces, blank pages, and SSRF URL guards.

### Prerequisites
- Notebooks 00–01

## 1. Setup
Load `.env` configuration via `course/utils/models.py` and initialize our shared LLM and embeddings helpers.

In [1]:
# If running in Google Colab, uncomment: !git clone https://github.com/rahat4528/rag-from-scratch-course-v2.git && %cd rag-from-scratch-course-v2 && pip install -q -r course/requirements.txt
import sys
from pathlib import Path
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course' / 'utils').exists()), Path.cwd())
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
from course.utils.models import get_llm, get_embeddings, resolve_course_root
course_dir = resolve_course_root(); data_dir = course_dir / 'data'
llm = get_llm(fallback_to_demo=True); embeddings = get_embeddings()
print(f'Ready | LLM={type(llm).__name__} | Embeddings={type(embeddings.base).__name__}')

[OfflineDemoChatModel] Notice: Provider 'ollama' not active (Ollama server is not reachable at http://localhost:11434. Install/start Ollama (`ollama pull llama3.2`) or switch RAG_PROVIDER in .env.). Running code-path demo mode.


W1112 00:42:35.880000 16580 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready | LLM=OfflineDemoChatModel | Embeddings=LocalSentenceTransformerEmbeddings


## 2. Concept: One Standard Envelope: `Document(page_content, metadata)`

A **Document Loader** is like a mail sorter converting postcards (`.txt`), glossy brochures (`.pdf`), and web articles (`.html`) into uniform envelopes: `Document(page_content=str, metadata=dict)`.

Using clean **native readers** (`pathlib`, `pypdf.PdfReader`, `BeautifulSoup`) avoids deprecated wrapper packages while keeping `.metadata` tidy (`source='policy_brief.pdf'`, NOT `C:\Users\YourName\...`).

```
.txt / .md  ---> Path.read_text(utf-8) --\
  .pdf        ---> pypdf.PdfReader --------+---> List[Document(page_content, metadata)]
  Folder      ---> loop & aggregate -------/
  Web URL     ---> SSRF Check + Soup -----/
```

## 3. Step-by-Step Build

### Step 1: Create & Inspect a LangChain `Document`
Notice a `Document` holds `.page_content` (string text) and `.metadata` (dictionary of tags such as source filename or page).

In [2]:
from langchain_core.documents import Document
from course.utils.helpers import load_text_document

doc_txt = load_text_document(data_dir / "company_handbook.txt", doc_type="handbook", access_level="employee")
print("Type   :", type(doc_txt).__name__)
print("Meta   :", doc_txt.metadata)
print("Preview:", doc_txt.page_content[:95].replace("\n", " "))
        

Type   : Document
Meta   : {'source': 'company_handbook.txt', 'doc_type': 'handbook', 'access_level': 'employee', 'doc_id': 'company_handbook.txt'}
Preview: NOVASPHERE DYNAMICS — EMPLOYEE & OPERATIONS HANDBOOK (2026 Edition) ===========================


> **What just happened?**
> Normalized metadata keeps source citations portable and concise.

### Step 2: Extract PDFs & Batch-Load an Entire Folder
Let's read `policy_brief.pdf` with our clean `pypdf`-backed helper, then load the whole `course/data/` folder in one call.

In [3]:
from course.utils.helpers import load_pdf_documents, load_folder_documents
pdf_pages = load_pdf_documents(data_dir / "policy_brief.pdf", doc_type="safety", access_level="executive")
folder_docs = load_folder_documents(data_dir)
print(f"PDF pages loaded: {len(pdf_pages)} | Chars: {len(pdf_pages[0].page_content)}")
for d in folder_docs:
    print(f" - {d.metadata['source']:<22} | type={d.metadata.get('doc_type')} | chars={len(d.page_content)}")
        

PDF pages loaded: 1 | Chars: 1169
 - company_handbook.txt   | type=txt | chars=2895
 - policy_brief.pdf       | type=pdf | chars=1169
 - product_faq.md         | type=md | chars=2071
 - recent_changelog.md    | type=md | chars=1163


> **What just happened?**
> `load_folder_documents` automatically skipped `eval_questions.json` cleanly while loading every `.txt`, `.md`, and `.pdf`!

### Step 3: Load Web Pages Safely (Offline Fallback + SSRF Guard)
Web loaders strip HTML tags (`<script>`, `<style>`) AND validate URLs against internal SSRF targets (`localhost`, `file://`, `169.254.*`).

In [4]:
from course.utils.helpers import load_web_page
offline_html = "<html><body><script>evil()</script><h1>Novasphere Docs</h1><p>gRPC v2 runs safely.</p></body></html>"
web_doc = load_web_page("https://example.invalid/docs", fallback_html=offline_html)
print("Clean web text:", web_doc.page_content, "| mode:", web_doc.metadata["fetch_mode"])
        

Clean web text: Novasphere Docs
gRPC v2 runs safely. | mode: offline_fallback


> **What just happened?**
> Providing `fallback_html` lets offline learners run web-ingestion lessons deterministically without flaky external servers.

### Step 4: Guard Common Traps (Empty Files, Corrupt PDFs & Unicode Spaces)
Let's test loading a blank file (`None`), an international UTF-8 filename with spaces, and messy PDF whitespace normalization.

In [5]:
import tempfile
with tempfile.TemporaryDirectory() as tmp:
    empty_p = Path(tmp) / "empty.txt"; empty_p.write_text("   ", encoding="utf-8")
    uni_p = Path(tmp) / "নিরাপত্তা brief (2026).txt"; uni_p.write_text("নিরাপত্তা — Safety first!", encoding="utf-8")
    assert load_text_document(empty_p) is None
    uni_doc = load_text_document(uni_p)
    print("Empty handled cleanly (None) | Unicode loaded:", uni_doc.metadata["source"], "->", uni_doc.page_content)
        

Empty handled cleanly (None) | Unicode loaded: নিরাপত্তা brief (2026).txt -> নিরাপত্তা — Safety first!


> **What just happened?**
> Empty files return `None` so blank chunks never pollute your vector database!

## 4. Try It Yourself
1. **Test SSRF URL protection** — *Hint:* Verify calling `load_web_page('http://127.0.0.1:8000/secret')` raises a `ValueError`.
2. **Normalize messy PDF newlines** — *Hint:* Replace triple spaces/newlines inside `pdf_pages[0].page_content` using `' '.join(text.split())`.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>
Run the cell below to check one working solution.
</details>

In [6]:
try:
    load_web_page("http://127.0.0.1:8000/secret")
except ValueError as err:
    print("SSRF blocked:", err)
cleaned = " ".join(pdf_pages[0].page_content.split())
print("Normalized PDF preview:", cleaned[:100])
        

SSRF blocked: Blocked non-public IP address '127.0.0.1' (SSRF defense).
Normalized PDF preview: Novasphere Dynamics - Autonomous Robotics Safety & Deployment Brief Document ID: SAF-2026-B | Classi


## 5. Common Mistakes
- Omitting `encoding='utf-8'` on Windows — crashes with `UnicodeDecodeError` on curly quotes or non-Latin scripts.
- Leaving machine-specific absolute paths in `.metadata['source']`.
- Allowing arbitrary user-entered URLs without blocking `file://` or `localhost` (SSRF vulnerability).

## 6. Recap
- LangChain `Document` standardizes raw files under `.page_content` and `.metadata`.
- Native `pathlib` + `pypdf` + `BeautifulSoup` load `.txt`, `.md`, `.pdf`, and HTML with zero deprecated warnings.
- Skipping empty files and sanitizing metadata prevents downstream chunk & security bugs.

## 7. Next Notebook
Next (`03_splitting_text.ipynb`): Slice long documents into overlapping chunks and measure how chunk size (100 vs 300 vs 800) affects search!

## 8. Video Chapter Outline
1. **Universal `Document` Container** — Page content + metadata catalog card.
2. **Text, Markdown & PDF Ingestion** — Native zero-warning Python loaders.
3. **Folder Batch Loading** — Handling mixed formats cleanly.
4. **SSRF-Safe Web Loading** — Stripping HTML noise + offline fallback.
5. **Edge Cases: Empty, Messy & Unicode** — Real-world hygiene checks.